In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from bhtrace.geometry import Photon, Observer
import bhtrace.geometry.spacetime as st
from bhtrace.tracing import PTracer

import bhtrace.medium as bhM
import bhtrace.graphics as bhg
import bhtrace.physics.electrodynamics as bhE
import bhtrace.utils.units as bhU

from bhtrace.grrt.runner import GRRT
from bhtrace.grrt.radiation import Blackbody, BolometricFlux
from bhtrace.utils.routines import xz_grid_4d_spherical

In [ ]:
# --- Configuration ---

N = 128  # Image resolution
D = 32
KERR_A = 0.5
OBS_R = 20.0
OBS_INCLINATION = torch.pi / 180 * 84
TRACE_T = 40
TRACE_NSTEPS = 512
TRACE_R_MAX = 256
TRACE_EPS = 1e-5
MASS = 2.2

system = bhU.NaturalGeometric(M=MASS)

base = st.KerrBL(a=KERR_A)
model = bhE.EulerHeisenberg(system, scale=100)
field = bhE.fields.SplitMonopole(B0=bhU.schwinger_B.to(system).f)
eff = st.EffectiveGeometry(base, model, field)

SPACETIMES = {
    'base': base,
    'eh_eff': st.EffectiveGeometry(base, model, field),
}

In [ ]:
# Checking the scale of the effective geometry
x = xz_grid_4d_spherical(N_x=4, x_lims=(4.0, 10.0))
eff_loc = eff.local(x)
print(eff_loc.ginv - eff_loc.base_loc.ginv)

In [ ]:
# Ray-tracing 
photonA = Photon(spacetime=base)
obsA = Observer(
    spacetime=base,
    r=OBS_R,
    inclination=OBS_INCLINATION,
)
obsA.generate_net(net_shape="square", net_rng=(N, N), net_size=(D, D))

photonB = Photon(spacetime=eff)
obsB = Observer(
    spacetime=eff,
    r=OBS_R,
    inclination=OBS_INCLINATION,
)
obsB.generate_net(net_shape="square", net_rng=(N, N), net_size=(D, D))


# Tracer configuration 
tracer = PTracer(ode_method="VCABM4")
tracer.__const_dx__ = True
device = "cuda" if torch.cuda.is_available() else 'cpu'
trace_params = {
    'T': TRACE_T, 'nsteps': TRACE_NSTEPS, 'r_max': TRACE_R_MAX,
    'eps': TRACE_EPS, 'device': device, 'dtype': torch.float64,
}
print(f"Using device: {trace_params['device']}")


# Running
X0, P0 = obsA.setup_ic(photonA)
traj_kerr = tracer.forward(
    photonA, X0, P0, **trace_params
)

X0, P0 = obsB.setup_ic(photonB)
traj_eff = tracer.forward(
    photonB, X0, P0, **trace_params
)

## Calculating Bolometric images

In [ ]:
# GRRT setup and computation
params = {'r_cut': 30, 'alpha': 0.9, 'mass_dot': 1e-12, 'clockwise': True}
medium = bhM.KeplerianDisk(base, MASS, **params) 

modelA = BolometricFlux()
modelB = BolometricFlux()

grrtA = GRRT(medium=medium)
grrtA.set_models(modelA)

grrtB = GRRT(medium=medium)
grrtB.set_models(modelB)

shoot_kerr = grrtA.compute(traj_kerr, dtype=torch.float64, device='cuda')
shoot_eff = grrtB.compute(traj_eff, dtype=torch.float64, device='cuda')

In [ ]:
# Visualisation 

image_kerr = modelA.intensities.x.cpu()
image_eff = modelB.intensities.x.cpu()

vmin = min(image_kerr.min(), image_eff.min())
vmax = max(image_kerr.max(), image_eff.max())

fig, axs = plt.subplots(1, 2, figsize=(30, 15))

im_kerr = image_kerr.view(N, N).cpu().transpose(-1, -2)
im_eff = image_eff.view(N, N).cpu().transpose(-1, -2)

imA = axs[0].imshow(im_kerr, cmap="afmhot", vmin=vmin, vmax=vmax)
axs[0].set_title(f"Kerr BH (a={KERR_A:.3f}) with KeplerianDisk")
imB = axs[1].imshow(im_eff, cmap="afmhot", vmin=vmin, vmax=vmax)
axs[1].set_title(f"Kerr BH (a={KERR_A:.3f}) with KeplerianDisk (NED-EH)")
fig.colorbar(imA, ax=axs[0])
fig.colorbar(imB, ax=axs[1])
plt.show()

## Calculating spectral images

In [ ]:
frequences = torch.logspace(1, 9, 32)

s_modelA = Blackbody(frequences)
s_modelB = Blackbody(frequences)

grrtA = GRRT(medium=medium)
grrtA.set_models(s_modelA)

grrtB = GRRT(medium=medium)
grrtB.set_models(s_modelB)

shoot_kerr_s = grrtA.compute(traj_kerr, dtype=torch.float64, device='cuda')
shoot_eff_s = grrtB.compute(traj_eff, dtype=torch.float64, device='cuda')

In [ ]:
# === Plot aggregated image
flux_spectral_A = s_modelA.intensities.x.cpu()
flux_spectral_B = s_modelB.intensities.x.cpu()

agg_image_A = flux_spectral_A.sum(dim=-1)
agg_image_B = flux_spectral_B.sum(dim=-1)

vmin = min(agg_image_A.min(), agg_image_B.min())
vmax = max(agg_image_A.max(), agg_image_B.max())

fig, axs = plt.subplots(1, 2, figsize=(16, 8))

im_kerr = agg_image_A.view(N, N).transpose(-1, -2)
im_eff = agg_image_B.view(N, N).transpose(-1, -2)

imA = axs[0].imshow(im_kerr, cmap="afmhot", vmin=vmin, vmax=vmax)
axs[0].set_title(f"Kerr BH (a={KERR_A:.3f}) with KeplerianDisk")
imB = axs[1].imshow(im_eff, cmap="afmhot", vmin=vmin, vmax=vmax)
axs[1].set_title(f"Kerr BH (a={KERR_A:.3f}) with KeplerianDisk (NED-EH)")
fig.colorbar(imA, ax=axs[0])
fig.colorbar(imB, ax=axs[1])
plt.show()

In [ ]:
# === Plot spectrum ===

spectrum_A = flux_spectral_A.nan_to_num(0.0).sum(dim=0)
spectrum_B = flux_spectral_B.nan_to_num(0.0).sum(dim=0)

fig, ax = plt.subplots(1, 1, figsize=(10, 8))

log_nu = frequences.log10().numpy() 
log_IA = spectrum_A.log10().numpy()
log_IB = spectrum_B.log10().numpy()

excepted = log_IA[0] + 4/3 * (log_nu - log_nu[0])

ax.plot(log_nu, log_IA, '-o')
ax.plot(log_nu, log_IB, '-x')
ax.set_xlabel('log10_nu')
ax.set_ylabel('log10_flux')
ax.grid('on')
ax.legend(['kerr', 'effective'])

plt.show()